# Camera Calibration Evaluation

In [ ]:
import json
from analysis.data_preprocessing import *
from analysis.camera_calibration import *
from segmentation.segmenter import *
from analysis.camera_calibration import * 
from segmentation.segmenter import *
from PIL import Image
import pandas as pd
import os
from tqdm.notebook import tqdm
import math

### Data and Experiment Setup

In [ ]:
chip_region = 3
EXPERIMENT_CONFIG_CHIP2 = {
    "data_dir": "../../data/Raw/Chip2-NaCl 0-19_",           # str, Image directory
    "image_batch": f"290-4-0.75x-{chip_region}",             # str, Specific image batch, e.g. 290-4-0.75x-3
    "zoom": 0.75,                                            # float, Camera zoom
    "cc": True,                                              # bool, Use camera calibration
    "ffc": False,                                             # bool, Use flatfield correction
    "clahe": False,                                          # bool, Use clahe (for segmentation only)
    "model": "CellposeSAM",                                  # str, SAM2 or CellposeSAM
    "approx": True,                                          # bool, Use approximated masks
    "approx_method": "ellipse",                              # str, "circle" or "ellipse"
    "shrinking_factor": {                                    # dict[str, int], scaling factor applied to approximated masks
        "large": 0.4, # 0.85,
        "medium": 0.4, # 0.7,
        "small": 0.4, # 0.55,
    },
    "overlay_masks": False,                                  # bool, use masks from first image on all following images
    "individual_intensities": False,                         # bool, show individual intensity points on intensity plots and link to segmentation results
    "filter_config": {                                       # dict, contains various configuration parameters for the segmentation mask filter
        "circularity_threshold": 0.7,                        # int, the circularity score used to filter irregular masks
        "border_thres": 2,                                   # int, the min number of pixels a mask has to be away from the border
        "max_groups": 3,                                     # int (2 or 3), the number of well sizes (2 if chip region 
        "eps_area": 40, 
        "eps_coords_medium": 90,
        "eps_coords_small": 70,
        "eps_coords_small_exclude": 20,
        "rough_only": False,
        "debug": False
    },
    "calibrator_config": {
        "ref_radii": {
            "region1": 500, # use large wells
            "region2": 175, # use medium wells - why is 250 too large?
            "region3": 500 # use large wells
        },
        "schematic_path1": "../../data/Raw/Chip-schematic/chip_cleaned_1_10.jpg",
        "schematic_path2": "../../data/Raw/Chip-schematic/chip_cleaned_2_8.jpg",
        "schematic_path3": "../../data/Raw/Chip-schematic/chip_cleaned_3_10.jpg",
        "calculate_individual": False,
        "chip_region": chip_region
    },
    "debug": True
}


EXPERIMENT_CONFIG = EXPERIMENT_CONFIG_CHIP2

# Collect the right images from the specified directory and group them in batches of increasing NaCl concentration
data = data_preprocessing(EXPERIMENT_CONFIG["data_dir"], EXPERIMENT_CONFIG["image_batch"])

# Isolate the specified batch as a list of dictionaries
data = data[EXPERIMENT_CONFIG["image_batch"] + ".jpg"]

print(json.dumps(data, indent=4))

### Object Setup

In [ ]:
# Camera
camera = Camera(
    zoom=EXPERIMENT_CONFIG["zoom"],
    um_per_pixel=6.45
)

# Segmenter
cellposeSAM = CellposeSAMWrapper()
segmenter = Segmenter(model=cellposeSAM, filter_config=EXPERIMENT_CONFIG["filter_config"])

# Calibrator
setup_image_path = data[0]["file_path"]
setup_image = np.array(Image.open(setup_image_path).convert('L'))

schematic_path1 = EXPERIMENT_CONFIG["calibrator_config"]["schematic_path1"] 
schematic_raw1 = np.array(Image.open(schematic_path1).convert('L'))
schematic_mask1 = extract_schematic(image=schematic_raw1, debug=EXPERIMENT_CONFIG["debug"])
schematic_path2 = EXPERIMENT_CONFIG["calibrator_config"]["schematic_path2"] 
schematic_raw2 = np.array(Image.open(schematic_path2).convert('L'))
schematic_mask2 = extract_schematic(image=schematic_raw2, debug=EXPERIMENT_CONFIG["debug"])
schematic_path3 = EXPERIMENT_CONFIG["calibrator_config"]["schematic_path3"] 
schematic_raw3 = np.array(Image.open(schematic_path3).convert('L'))
schematic_mask3 = extract_schematic(image=schematic_raw3, debug=EXPERIMENT_CONFIG["debug"])
ref_images = {
    "region1": schematic_mask1,
    "region2": schematic_mask2,
    "region3": schematic_mask3,
}

calibrator = CameraCalibrator(
    setup_image=setup_image,
    camera=camera,
    segmenter=segmenter,
    ref_images=ref_images,
    ref_radii=EXPERIMENT_CONFIG["calibrator_config"]["ref_radii"],
    calculate_individual=EXPERIMENT_CONFIG["calibrator_config"]["calculate_individual"]
)

### Run Experiment

In [ ]:
df = pd.DataFrame(columns=["iou_all", "iou_large", "iou_medium", "iou_small"])

for image_data in tqdm(data):
    name = image_data["file_path"].split('/')[-1].split('.')[0]
    print(f"==== Evaluating camera calibration for image {name}. ====")
    
    # Open Image
    gray_image = np.array(Image.open(image_data["file_path"]).convert("L"))

    corrected, affine, _ = calibrator.undistort(
        gray_image,
        chip_region=EXPERIMENT_CONFIG['calibrator_config']["chip_region"],
        debug=EXPERIMENT_CONFIG["debug"]
    )
    
    plt.figure(figsize=(10,10))
    plt.imshow(corrected, cmap='gray')
    plt.show()

    iou_all, iou_large, iou_medium, iou_small = evaluate_camera_calibration(
        image=affine, 
        ref_image=calibrator.ref_image, 
        segmenter=segmenter, 
        debug=EXPERIMENT_CONFIG["debug"]
    )
    df.loc[len(df)] = [iou_all, iou_large, iou_medium, iou_small]

    iou_all, iou_large, iou_medium, iou_small = evaluate_camera_calibration(
        image=corrected, 
        ref_image=calibrator.ref_image, 
        segmenter=segmenter, 
        debug=EXPERIMENT_CONFIG["debug"]
    )
    df.loc[len(df)] = [iou_all, iou_large, iou_medium, iou_small]


In [ ]:
print(df)

In [ ]:
import pandas as pd
import numpy as np

import pandas as pd

def save_df_with_even_odd_averages(df, filename='output.csv'):
    # Select even and odd rows by index
    even_rows = df.iloc[::2]   # rows 0, 2, 4, ...
    odd_rows = df.iloc[1::2]   # rows 1, 3, 5, ...

    # Compute averages (numeric columns only)
    even_avg = even_rows.mean(numeric_only=True)
    odd_avg = odd_rows.mean(numeric_only=True)

    # Label the rows
    even_avg.name = 'average_even_rows'
    odd_avg.name = 'average_odd_rows'

    # Print the averages
    print("Average of even-indexed rows:")
    print(even_avg)
    print("Average of odd-indexed rows:")
    print(odd_avg)

    # Append to DataFrame
    df_with_averages = pd.concat([df, even_avg.to_frame().T, odd_avg.to_frame().T], axis=0)

    # Save to CSV
    df_with_averages.to_csv(filename, index=True)

    return df_with_averages

result = save_df_with_even_odd_averages(df)